<a href="https://colab.research.google.com/github/beingAnujChaudhary/Customer-Churn-Prediction-Retention-Prioritization/blob/main/notebooks/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# %% [markdown]
# # 📊 Customer Churn EDA + Preprocessing
# *Notebook: 01_eda.ipynb | Stage 1-3 Complete*
#
# **Author**: Anuj Chaudhary
# **Purpose**: Load data, validate, explore, preprocess, and prepare for modeling

# %% [code]
# INSTALL DEPENDENCIES (Colab only - skip if running locally with venv)
!pip install -q pandas==2.0.3 numpy==1.24.3 matplotlib==3.7.2 seaborn==0.13.0 plotly==5.17.0 scikit-learn==1.3.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 14.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [2]:
# %% [code]
# IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Plot settings
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
%matplotlib inline

print("✓ Libraries loaded successfully")

✓ Libraries loaded successfully


In [3]:
from google.colab import files
uploaded = files.upload()  # Upload BankChurners.csv
df = pd.read_csv("BankChurners.csv")

# Drop Kaggle artifacts
if df.columns[-2:].str.contains('Naive_Bayes_Classifier').any():
    df = df.iloc[:, :-2]

print(f"✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Saving BankChurners.csv to BankChurners.csv
✅ Dataset loaded: 10,127 rows × 21 columns


In [4]:
# %% [code]
# DATA VALIDATION CHECKS
# 1. Target column exists
if 'Attrition_Flag' not in df.columns:
    raise ValueError("❌ Attrition_Flag column missing - wrong dataset?")

# 2. Handle empty strings in categorical columns
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    empty_count = (df[col] == '').sum()
    if empty_count > 0:
        print(f"⚠️ {col}: {empty_count} empty strings → replacing with 'Unknown'")
        df[col] = df[col].replace('', 'Unknown')

# 3. Check for unexpected nulls in key features
key_features = ['Credit_Limit', 'Total_Trans_Ct', 'Months_Inactive_12_mon']
for col in key_features:
    if df[col].isnull().any():
        print(f"⚠️ {col}: {df[col].isnull().sum()} null values found")

print("✅ Data validation complete")

✅ Data validation complete


In [5]:
# %% [code]
print("🔍 DATA QUALITY CHECK")
print(f"\n1. Shape: {df.shape}")

missing = df.isnull().sum()
if missing.sum() > 0:
    print(f"\n2. Missing Values:\n{missing[missing > 0]}")
else:
    print(f"\n2. Missing Values: None ✓")

print(f"\n3. Target Distribution (Attrition_Flag):")
print(df['Attrition_Flag'].value_counts(normalize=True).map('{:.2%}'.format))

print(f"\n4. Feature Types:\n{df.dtypes.value_counts()}")

print(f"\n5. Numeric Summary (key columns):")
numeric_cols = ['Customer_Age', 'Credit_Limit', 'Total_Trans_Ct', 'Total_Revolving_Bal']
print(df[numeric_cols].describe().T[['mean', 'std', 'min', 'max']])

🔍 DATA QUALITY CHECK

1. Shape: (10127, 21)

2. Missing Values: None ✓

3. Target Distribution (Attrition_Flag):
Attrition_Flag
Existing Customer    83.93%
Attrited Customer    16.07%
Name: proportion, dtype: object

4. Feature Types:
int64      10
object      6
float64     5
Name: count, dtype: int64

5. Numeric Summary (key columns):
                            mean          std     min      max
Customer_Age           46.325960     8.016814    26.0     73.0
Credit_Limit         8631.953698  9088.776650  1438.3  34516.0
Total_Trans_Ct         64.858695    23.472570    10.0    139.0
Total_Revolving_Bal  1162.814061   814.987335     0.0   2517.0


In [6]:
# %% [code]
# Convert target to binary for modeling
df["Churn"] = df["Attrition_Flag"].map({
    "Existing Customer": 0,
    "Attrited Customer": 1
})

# Verify conversion
print(f"✅ Target converted: {df['Churn'].value_counts().to_dict()}")
df[['Attrition_Flag', 'Churn']].head()

✅ Target converted: {0: 8500, 1: 1627}


,Attrition_Flag,Churn
0,Existing Customer,0
1,Existing Customer,0
2,Existing Customer,0
3,Existing Customer,0
4,Existing Customer,0


In [7]:
# %% [code]
# Create output directory for plots
os.makedirs("output/plots", exist_ok=True)
print("✅ output/plots directory ready")

# Key behavioural columns for analysis
behavioural_cols = [
    "Total_Trans_Ct",          # Transaction count
    "Months_Inactive_12_mon",  # Inactivity
    "Contacts_Count_12_mon",   # Support contacts
    "Credit_Limit",
    "Total_Relationship_Count" # Products held
]

# Generate and save boxplots
for col in behavioural_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x="Churn", y=col, palette="Set2")
    plt.title(f"{col} by Churn Status", fontsize=13, fontweight='bold')
    plt.xlabel("Churn (1=Yes, 0=No)")
    plt.tight_layout()

    # Save high-res for portfolio
    plt.savefig(f"output/plots/{col}_by_churn.png", dpi=300, bbox_inches='tight')
    plt.close()  # Free memory
    print(f"✅ Saved: output/plots/{col}_by_churn.png")

✅ output/plots directory ready


/tmp/ipykernel_27895/1036030098.py:18: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="Churn", y=col, palette="Set2")


✅ Saved: output/plots/Total_Trans_Ct_by_churn.png


/tmp/ipykernel_27895/1036030098.py:18: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="Churn", y=col, palette="Set2")


✅ Saved: output/plots/Months_Inactive_12_mon_by_churn.png


...

✅ Saved: output/plots/Contacts_Count_12_mon_by_churn.png


/tmp/ipykernel_27895/1036030098.py:18: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="Churn", y=col, palette="Set2")


✅ Saved: output/plots/Credit_Limit_by_churn.png


/tmp/ipykernel_27895/1036030098.py:18: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="Churn", y=col, palette="Set2")


✅ Saved: output/plots/Total_Relationship_Count_by_churn.png


In [8]:
# %% [code]
# Drop ID & artifact columns
cols_to_drop = ["CLIENTNUM", "Attrition_Flag"]  # Drop original target, keep binary 'Churn'
df_clean = df.drop(columns=cols_to_drop, errors="ignore")
print(f"✅ Dropped {len(cols_to_drop)} columns. Remaining: {df_clean.shape[1]} features")

✅ Dropped 2 columns. Remaining: 20 features


In [10]:
# %% [code]
# Encode categorical variables
cat_cols = ["Gender", "Education_Level", "Marital_Status", "Income_Category", "Card_Category"]

le_dict = {}  # Store encoders for later reference
for col in cat_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    le_dict[col] = le
    print(f"✅ Encoded '{col}': {dict(zip(le.classes_, range(len(le.classes_))))}")

print("\n✅ All categorical variables encoded")

✅ Encoded 'Gender': {np.int64(0): 0, np.int64(1): 1}
✅ Encoded 'Education_Level': {np.int64(0): 0, np.int64(1): 1, np.int64(2): 2, np.int64(3): 3, np.int64(4): 4, np.int64(5): 5, np.int64(6): 6}
✅ Encoded 'Marital_Status': {np.int64(0): 0, np.int64(1): 1, np.int64(2): 2, np.int64(3): 3}
✅ Encoded 'Income_Category': {np.int64(0): 0, np.int64(1): 1, np.int64(2): 2, np.int64(3): 3, np.int64(4): 4, np.int64(5): 5}
✅ Encoded 'Card_Category': {np.int64(0): 0, np.int64(1): 1, np.int64(2): 2, np.int64(3): 3}

✅ All categorical variables encoded


In [11]:
# %% [code]
# Separate features & target
X = df_clean.drop(columns=["Churn"])
y = df_clean["Churn"]

# Stratified split (preserves 16% churn ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"✅ Split complete:")
print(f"   Train: {X_train.shape[0]} rows | Churn rate: {y_train.mean():.2%}")
print(f"   Test:  {X_test.shape[0]} rows  | Churn rate: {y_test.mean():.2%}")

✅ Split complete:
   Train: 8101 rows | Churn rate: 16.07%
   Test:  2026 rows  | Churn rate: 16.04%


In [12]:
# %% [code]
# Combine for saving with split indicator
train_df = X_train.copy()
train_df["Churn"] = y_train
train_df["split"] = "train"

test_df = X_test.copy()
test_df["Churn"] = y_test
test_df["split"] = "test"

full_processed = pd.concat([train_df, test_df], ignore_index=True)

# Save with versioning
os.makedirs("data/processed", exist_ok=True)
full_processed.to_csv("data/processed/customers_v2.csv", index=False)
print("✅ Saved: data/processed/customers_v2.csv")

✅ Saved: data/processed/customers_v2.csv


In [13]:
# %% [code]
# Critical validation checks
assert full_processed.isnull().sum().sum() == 0, "❌ Missing values detected!"
assert "Churn" in full_processed.columns, "❌ 'Churn' column missing!"
assert set(full_processed["Churn"].unique()) == {0, 1}, "❌ Target must be binary!"
assert "split" in full_processed.columns, "❌ 'split' column missing!"

print("🎉 All preprocessing validations passed!")

🎉 All preprocessing validations passed!
